<a href="https://colab.research.google.com/github/1908hanjm/walmart-sales-analysis/blob/main/notebooks/03_sql_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. SQL Validation — Database Build & Cross-Validation

📌 **Executive Summary**
To prove the analysis is tool-agnostic and reproducible, this notebook builds a **SQLite database** from the cleaned dataset and re-runs the three EDA questions (Q1–Q3) in **SQL**. Every SQL result is cross-validated against the equivalent **pandas** computation.

**Validation Scope:**
1. **Database Build**: Load `clean_walmart_data.csv` (421,570 rows) into SQLite table `weekly_sales` with row-count verification.
2. **Q1 — Yearly sales totals**: SQL `GROUP BY` year vs pandas `groupby`.
3. **Q2 — Store-type performance**: SQL aggregates by `Type` vs pandas `groupby`.
4. **Q3 — Holiday lift**: SQL conditional `AVG` on `IsHoliday` vs pandas `groupby`.


### Step 1: Build the SQLite Database
Loading the cleaned dataset and creating a SQLite database (`walmart_sales.db`) with a single `weekly_sales` table. Row counts are verified after loading.

In [1]:
# ==========================================
# 1. Build SQLite Database from Cleaned CSV
# ==========================================
from google.colab import drive
import pandas as pd
import numpy as np
import sqlite3
import os

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/포트폴리오-walmart(캐글)/'
CSV_PATH = DATA_PATH + 'clean_walmart_data.csv'
DB_PATH = DATA_PATH + 'walmart_sales.db'

df = pd.read_csv(CSV_PATH)
print(f"Loaded cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Normalize types for SQL compatibility
df['Date'] = pd.to_datetime(df['Date'])
df['IsHoliday'] = df['IsHoliday'].astype(str).str.lower().isin(['true', '1'])

# (Re)build the SQLite database
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH)
df.to_sql('weekly_sales', conn, if_exists='replace', index=False)

db_rows = pd.read_sql("SELECT COUNT(*) AS n FROM weekly_sales", conn)['n'][0]
print(f"Rows in SQLite table 'weekly_sales': {db_rows:,}")
assert db_rows == len(df), "Row count mismatch between CSV and SQLite!"
print("SUCCESS: database built, row counts match.")

Mounted at /content/drive
Loaded cleaned data: 421,570 rows x 19 columns
Rows in SQLite table 'weekly_sales': 421,570
SUCCESS: database built, row counts match.


### Step 2: Q1 Cross-Validation — Yearly Sales Totals
Recomputing total sales per year in SQL (`GROUP BY` on the year extracted from `Date`) and comparing against the pandas `groupby` result.

In [2]:
# ==========================================
# 2. Q1 Validation: Yearly Sales Totals
# ==========================================
q1_sql = pd.read_sql("""
    SELECT SUBSTR(Date, 1, 4) AS Year,
           SUM(Weekly_Sales)  AS Total_Sales,
           COUNT(*)           AS Records
    FROM weekly_sales
    GROUP BY Year
    ORDER BY Year
""", conn)

q1_pd = (df.assign(Year=df['Date'].dt.year.astype(str))
           .groupby('Year', as_index=False)
           .agg(Total_Sales=('Weekly_Sales', 'sum'),
                Records=('Weekly_Sales', 'size'))
           .sort_values('Year'))

print("--- SQL ---")
print(q1_sql.to_string(index=False))
print("\n--- pandas ---")
print(q1_pd.to_string(index=False))

assert np.allclose(q1_sql['Total_Sales'], q1_pd['Total_Sales']), "Q1 totals mismatch!"
assert (q1_sql['Records'].values == q1_pd['Records'].values).all(), "Q1 record counts mismatch!"
print("\nSUCCESS: Q1 validated \u2014 SQL and pandas yearly totals match.")

--- SQL ---
Year  Total_Sales  Records
2010 2.288886e+09   140679
2011 2.448200e+09   153453
2012 2.000133e+09   127438

--- pandas ---
Year  Total_Sales  Records
2010 2.288886e+09   140679
2011 2.448200e+09   153453
2012 2.000133e+09   127438

SUCCESS: Q1 validated — SQL and pandas yearly totals match.


### Step 3: Q2 Cross-Validation — Store-Type Performance
Recomputing average weekly sales, total sales, average store size, and store counts per store type in SQL and comparing against pandas.

In [3]:
# ==========================================
# 3. Q2 Validation: Performance by Store Type
# ==========================================
q2_sql = pd.read_sql("""
    SELECT Type,
           AVG(Weekly_Sales)     AS Avg_Weekly_Sales,
           SUM(Weekly_Sales)     AS Total_Sales,
           AVG(Size)             AS Avg_Store_Size,
           COUNT(DISTINCT Store) AS Store_Count
    FROM weekly_sales
    GROUP BY Type
    ORDER BY Type
""", conn)

q2_pd = (df.groupby('Type', as_index=False)
           .agg(Avg_Weekly_Sales=('Weekly_Sales', 'mean'),
                Total_Sales=('Weekly_Sales', 'sum'),
                Avg_Store_Size=('Size', 'mean'),
                Store_Count=('Store', 'nunique'))
           .sort_values('Type'))

print("--- SQL ---")
print(q2_sql.round(2).to_string(index=False))
print("\n--- pandas ---")
print(q2_pd.round(2).to_string(index=False))

for col in ['Avg_Weekly_Sales', 'Total_Sales', 'Avg_Store_Size']:
    assert np.allclose(q2_sql[col], q2_pd[col]), f"Q2 mismatch in {col}!"
assert (q2_sql['Store_Count'].values == q2_pd['Store_Count'].values).all(), "Q2 store counts mismatch!"
print("\nSUCCESS: Q2 validated \u2014 SQL and pandas store-type metrics match.")

--- SQL ---
Type  Avg_Weekly_Sales  Total_Sales  Avg_Store_Size  Store_Count
   A          20099.57 4.331015e+09       182231.29           22
   B          12237.08 2.000701e+09       101818.74           17
   C           9519.53 4.055035e+08        40535.73            6

--- pandas ---
Type  Avg_Weekly_Sales  Total_Sales  Avg_Store_Size  Store_Count
   A          20099.57 4.331015e+09       182231.29           22
   B          12237.08 2.000701e+09       101818.74           17
   C           9519.53 4.055035e+08        40535.73            6

SUCCESS: Q2 validated — SQL and pandas store-type metrics match.


### Step 4: Q3 Cross-Validation — Holiday Sales Lift
Recomputing average weekly sales for holiday vs non-holiday weeks in SQL (conditional aggregation on `IsHoliday`) and comparing against pandas. This reproduces the +7.13% holiday lift reported in notebook 02.

In [4]:
# ==========================================
# 4. Q3 Validation: Holiday vs Non-Holiday Lift
# ==========================================
q3_sql = pd.read_sql("""
    SELECT AVG(CASE WHEN IsHoliday = 1 THEN Weekly_Sales END) AS Holiday_Avg,
           AVG(CASE WHEN IsHoliday = 0 THEN Weekly_Sales END) AS Non_Holiday_Avg
    FROM weekly_sales
""", conn).iloc[0]

q3_pd = df.groupby('IsHoliday')['Weekly_Sales'].mean()
hol_sql, non_sql = q3_sql['Holiday_Avg'], q3_sql['Non_Holiday_Avg']
hol_pd, non_pd = q3_pd.loc[True], q3_pd.loc[False]
lift_pct = (hol_sql - non_sql) / non_sql * 100

print(f"SQL    : holiday=${hol_sql:,.2f} | non-holiday=${non_sql:,.2f}")
print(f"pandas : holiday=${hol_pd:,.2f} | non-holiday=${non_pd:,.2f}")
print(f"Holiday sales lift: +{lift_pct:.2f}%")

assert np.isclose(hol_sql, hol_pd) and np.isclose(non_sql, non_pd), "Q3 mismatch!"
print("SUCCESS: Q3 validated \u2014 SQL and pandas holiday lifts match.")
conn.close()

SQL    : holiday=$17,035.82 | non-holiday=$15,901.45
pandas : holiday=$17,035.82 | non-holiday=$15,901.45
Holiday sales lift: +7.13%
SUCCESS: Q3 validated — SQL and pandas holiday lifts match.


## \u2705 Validation Summary

| Question | SQL | pandas | Match |
|---|---|---|---|
| Q1 \u2014 Yearly sales totals | `GROUP BY SUBSTR(Date,1,4)` | `groupby(Year)` | \u2705 |
| Q2 \u2014 Store-type performance | `GROUP BY Type` | `groupby(Type)` | \u2705 |
| Q3 \u2014 Holiday lift (+7.13%) | Conditional `AVG` | `groupby(IsHoliday)` | \u2705 |

All three business questions reproduce **identical results** in SQL and pandas, confirming the analysis is correct and tool-agnostic. The SQLite database (`walmart_sales.db`) is saved next to the cleaned CSV for reuse.